# Japonca Altyazı Pipeline (Whisper)

Video/ses dosyasından Japonca SRT oluştur.

**Pipeline:** `Video → ffmpeg → [Demucs vocal] → Whisper → [Timestamp fix] → [LLM düzeltme] → SRT`

## Config (B1 hücresinde)
| Parametre | Seçenekler | Default |
|-----------|-----------|--------|
| `WHISPER_MODEL` | `'large-v3'`, `'kotoba-tech/kotoba-whisper-v2.2'` | `'large-v3'` |
| `USE_VOICE_ISOLATION` | `True` / `False` | `True` |
| `TIMESTAMP_METHOD` | `'stable-ts'`, `'none'` | `'stable-ts'` |
| `USE_LLM_REFINEMENT` | `True` / `False` | `False` |

## Colab Secrets
- `HF_TOKEN` — HuggingFace
- `VLLM_API_KEY` — LLM düzeltme kullanırsan

## Kullanım
A: Kurulum → B: Config + dosya → C: Transcript → D: (opsiyonel) LLM düzeltme → E: SRT

---
# A) Kurulum (bir kere)

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
print(f'\u2705 GPU: {torch.cuda.get_device_name(0)}')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

print('\U0001f4e6 Paketler kuruluyor...')
!pip install -q faster-whisper stable-ts demucs httpx
!apt-get -qq install ffmpeg > /dev/null 2>&1

LOG_DIR = '/content/logs'
os.makedirs(LOG_DIR, exist_ok=True)
print('\u2705 Kurulum tamam')

---
# B) Config + Dosya

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIG — değiştir, C'den itibaren tekrar çalıştır         ║
# ╚══════════════════════════════════════════════════════════════╝

WHISPER_MODEL       = 'large-v3'     # 'large-v3' | 'RoachLin/kotoba-whisper-v2.2-faster'
USE_VOICE_ISOLATION = True           # Demucs ile müzik/SFX ayrıştırma
TIMESTAMP_METHOD    = 'stable-ts'    # 'stable-ts' | 'none'
USE_LLM_REFINEMENT  = False          # Harici LLM ile düzeltme

# LLM Düzeltme ayarları (USE_LLM_REFINEMENT = True ise)
LLM_API_URL  = 'https://api.ersamely.com/v1'  # OpenAI uyumlu herhangi bir endpoint
LLM_API_KEY  = ''                              # Colab Secrets'ten alınır (aşağıda)
LLM_MODEL    = 'Qwen/Qwen3.6-35B-A3B-FP8'    # veya 'gpt-4o', 'claude-3-sonnet', vs.

# API Key'i Colab Secrets'ten al
if USE_LLM_REFINEMENT:
    from google.colab import userdata
    LLM_API_KEY = userdata.get('VLLM_API_KEY')

print(f'Whisper:    {WHISPER_MODEL}')
print(f'Vocal izol: {USE_VOICE_ISOLATION}')
print(f'Timestamp:  {TIMESTAMP_METHOD}')
print(f'LLM düzelt: {USE_LLM_REFINEMENT}')
if USE_LLM_REFINEMENT:
    print(f'  → {LLM_API_URL} ({LLM_MODEL})')

In [ ]:
from google.colab import files

print('Dosya seç:')
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
print(f'\u2705 {INPUT_FILE}')

AUDIO_FILE = INPUT_FILE
if INPUT_FILE.lower().endswith(('.mp4', '.mkv', '.avi', '.webm', '.mov')):
    AUDIO_FILE = 'audio.wav'
    subprocess.run(
        ['ffmpeg', '-y', '-i', INPUT_FILE, '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', AUDIO_FILE],
        capture_output=True, check=True,
    )
    print(f'\u2705 Ses çıkarıldı')

In [ ]:
import time

TRANSCRIPT_INPUT = AUDIO_FILE

if USE_VOICE_ISOLATION:
    print('\U0001f3b5 Vocal izolasyon (Demucs htdemucs_ft)...')
    DEMUCS_INPUT = 'audio_44k.wav'
    subprocess.run(
        ['ffmpeg', '-y', '-i', AUDIO_FILE, '-ar', '44100', '-ac', '2', DEMUCS_INPUT],
        capture_output=True, check=True,
    )
    t0 = time.time()
    result = subprocess.run(
        ['python', '-m', 'demucs', '--two-stems', 'vocals', '-n', 'htdemucs_ft', DEMUCS_INPUT],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        stem = os.path.splitext(os.path.basename(DEMUCS_INPUT))[0]
        vocal_path = f'separated/htdemucs_ft/{stem}/vocals.wav'
        if os.path.exists(vocal_path):
            TRANSCRIPT_INPUT = vocal_path
            print(f'\u2705 Vocal izole ({time.time()-t0:.1f}s)')
        else:
            print('\u26a0\ufe0f Vocal dosyası bulunamadı, orijinal kullanılacak')
    else:
        print(f'\u26a0\ufe0f Demucs başarısız:\n{result.stderr[-500:]}')
else:
    print('Voice isolation kapalı')

---
# C) Transcript

In [ ]:
import time

segments = []

if TIMESTAMP_METHOD == 'stable-ts':
    import stable_whisper
    print(f'\U0001f4e5 stable-ts + {WHISPER_MODEL}')
    model = stable_whisper.load_faster_whisper(WHISPER_MODEL)
    t0 = time.time()
    result = model.transcribe_stable(
        TRANSCRIPT_INPUT, task='transcribe', language='ja', beam_size=5, vad=True,
    )
    for seg in result.segments:
        segments.append({'start': seg.start, 'end': seg.end, 'text': seg.text.strip()})
    print(f'\u2705 {len(segments)} segment, {time.time()-t0:.1f}s [stable-ts]')

else:
    from faster_whisper import WhisperModel
    print(f'\U0001f4e5 faster-whisper {WHISPER_MODEL}')
    model = WhisperModel(WHISPER_MODEL, device='cuda', compute_type='float16')
    t0 = time.time()
    segs, _ = model.transcribe(
        TRANSCRIPT_INPUT, task='transcribe', language='ja', beam_size=5,
        vad_filter=True, vad_parameters=dict(min_silence_duration_ms=300, speech_pad_ms=200),
    )
    for seg in segs:
        segments.append({'start': seg.start, 'end': seg.end, 'text': seg.text.strip()})
    print(f'\u2705 {len(segments)} segment, {time.time()-t0:.1f}s [ham]')

for s in segments[:10]:
    print(f'  [{s["start"]:7.2f}-{s["end"]:7.2f}] {s["text"]}')

# GPU serbest
del model
torch.cuda.empty_cache()
import gc; gc.collect()
print('\n\u2705 Model unload')

---
# D) LLM Düzeltme (opsiyonel)

`USE_LLM_REFINEMENT = False` ise bu bölümü atla.

In [ ]:
if not USE_LLM_REFINEMENT:
    refined_segments = [{'start': s['start'], 'end': s['end'], 'text': s['text'], 'refined': s['text']} for s in segments]
    print('LLM düzeltme kapalı — ham transcript kullanılacak')
else:
    import time
    import requests

    BATCH_SIZE = 20
    SYSTEM = """あなたは日本語の字幕校正の専門家です。音声認識が生成した字幕を校正してください。
ルール：漢字誤変換修正、句読点追加、文区切り修正、話し言葉の自然さ保持、意味を変えない、同じ行数・順番で番号付き出力。"""

    refined_segments = []
    total = (len(segments) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'\U0001f4dd LLM düzeltme: {LLM_API_URL} / {LLM_MODEL}')
    print(f'   {len(segments)} segment, {total} batch\n')
    t0 = time.time()

    for bi in range(total):
        batch = segments[bi*BATCH_SIZE:(bi+1)*BATCH_SIZE]
        jp_lines = '\n'.join(f'{i+1}. {s["text"]}' for i, s in enumerate(batch))
        try:
            r = requests.post(
                f'{LLM_API_URL}/chat/completions',
                headers={'Authorization': f'Bearer {LLM_API_KEY}'},
                json={'model': LLM_MODEL, 'messages': [
                    {'role': 'system', 'content': SYSTEM},
                    {'role': 'user', 'content': jp_lines},
                ], 'temperature': 0.2, 'max_tokens': 2048},
                timeout=120,
            )
            if r.status_code == 200:
                content = r.json()['choices'][0]['message'].get('content', '')
                lines = []
                for line in content.strip().split('\n'):
                    line = line.strip()
                    if not line: continue
                    if line[0].isdigit():
                        parts = line.split('.', 1) if '.' in line[:4] else line.split(')', 1)
                        if len(parts) == 2: line = parts[1].strip()
                    lines.append(line)
                for i, s in enumerate(batch):
                    refined_segments.append({**s, 'refined': lines[i] if i < len(lines) else s['text']})
                print(f'  \u2705 {bi+1}/{total}')
            else:
                for s in batch: refined_segments.append({**s, 'refined': s['text']})
                print(f'  \u274c {bi+1}: HTTP {r.status_code}')
        except requests.RequestException as e:
            for s in batch: refined_segments.append({**s, 'refined': s['text']})
            print(f'  \u274c {bi+1}: {e}')

    print(f'\n\u2705 Düzeltme tamamlandı ({time.time()-t0:.1f}s)')

---
# E) SRT İndir

In [ ]:
from pathlib import Path
from google.colab import files

# D atlandıysa segments'i doğrudan kullan
if 'refined_segments' not in dir():
    refined_segments = [{'start': s['start'], 'end': s['end'], 'text': s['text'], 'refined': s['text']} for s in segments]


def ts(sec):
    h, m, s, ms = int(sec//3600), int(sec%3600//60), int(sec%60), int(sec%1*1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'


stem = Path(INPUT_FILE).stem
model_tag = 'kotoba' if 'kotoba' in WHISPER_MODEL else 'v3'
ts_tag = f'_{TIMESTAMP_METHOD}' if TIMESTAMP_METHOD != 'none' else ''

# Ham transcript SRT
raw_srt = f'{stem}_ja_{model_tag}{ts_tag}.srt'
with open(raw_srt, 'w', encoding='utf-8') as f:
    for i, s in enumerate(refined_segments, 1):
        f.write(f'{i}\n{ts(s["start"])} --> {ts(s["end"])}\n{s["text"]}\n\n')

output_files = [raw_srt]
print(f'\u2705 {raw_srt}')

# LLM düzeltilmiş SRT (eğer farklıysa)
if USE_LLM_REFINEMENT:
    llm_srt = f'{stem}_ja_{model_tag}{ts_tag}_llm.srt'
    with open(llm_srt, 'w', encoding='utf-8') as f:
        for i, s in enumerate(refined_segments, 1):
            f.write(f'{i}\n{ts(s["start"])} --> {ts(s["end"])}\n{s["refined"]}\n\n')
    output_files.append(llm_srt)
    print(f'\u2705 {llm_srt} (LLM düzeltilmiş)')

# İndir
for f_path in output_files:
    files.download(f_path)
print(f'\U0001f4e6 {len(output_files)} dosya indirildi')